In [1]:
import pymupdf4llm
import ollama
import re
from pathlib import Path


In [4]:
def describe_image_with_ollama(image_path: str) -> str:
    """Envoie l'image à Ollama et retourne sa description."""
    
    system_prompt = """
    You are a precise document analyst. When describing images for text documents, follow these rules:
    1. Start with the image type (Graph, Schema, Diagram, Screenshot, Photo, Table, etc.)
    2. Provide a concise but complete description
    3. Focus on factual, structural elements
    4. Use neutral, professional language
    5. Keep descriptions to 2-3 sentences maximum

    Examples:
    - "[Graph]: A line chart comparing Bitcoin price evolution from 2020 to 2024, showing a sharp increase in 2021 followed by a correction period and gradual recovery."
    - "[Schema]: An organizational flowchart showing the company hierarchy with CEO at top, followed by CTO, CFO, and CMO divisions, each managing 3-4 department heads."
    - "[Table]: A data table with 4 columns listing transaction IDs, timestamps, amounts in BTC, and wallet addresses for the last 10 blockchain transactions."
    - "[Diagram]: A technical architecture diagram illustrating a 3-tier web application with load balancer, application servers, and database cluster."
    - "[Photo]: A screenshot of a cryptocurrency trading dashboard displaying real-time price charts, order book, and portfolio balance."
    - "[Schema]: A Venn diagram showing the overlap between machine learning, deep learning, and neural networks, with specific algorithms listed in each intersection."
"""
    # Assurez-vous d'avoir téléchargé un modèle vision, ex: 'llava' ou 'llama3.2-vision'
    response = ollama.chat(
        model='gemma4:31b-cloud',
        messages=[
            {
            'role': 'system',
            'content': system_prompt
        }, {
            'role': 'user',
            'content': "Give a brief description of the image so it can be included in a text document.",
            'images': [image_path] # Ollama gère directement les chemins de fichiers !
        }]
    )
    return response['message']['content']

def replace_with_description(match: re.Match) -> str:
    img_path = match.group(1)
    print(f"Analyse de l'image : {img_path} via Ollama...")
    
    try:
        description = describe_image_with_ollama(img_path)
        # On formate la description pour qu'elle se démarque dans le Markdown
        return f"""\n <AI img description>\n*{description.strip()}*\n</AI img description>\n"""
    except Exception as e:
        return f"\n> *[Erreur lors de l'analyse de l'image : {e}]*\n"

In [5]:
def pdf_to_semantic_markdown(pdf_path: str, output_md: str, img_dir: str = "images_tmp"):
    Path(img_dir).mkdir(exist_ok=True)
    
    # 1. Extraction avec sauvegarde physique des images
    print("Extraction du PDF...")
    md_text = pymupdf4llm.to_markdown(
        pdf_path,
        write_images=True,
        image_path=img_dir,
        image_format="png",
        force_text=False
    )
    
    # 2. Expression régulière pour trouver les images Markdown: ![alt](chemin)
    img_pattern = re.compile(r'!\[.*?\]\((.*?)\)')
    
    # # 3. Remplacement de toutes les balises images par leur description
    # print("Substitution des images par le texte...")
    final_md_text = img_pattern.sub(replace_with_description, md_text)
    
    # 4. Sauvegarde du fichier final
    Path(output_md).write_text(final_md_text, encoding="utf-8")

    print("Terminé !")

# Utilisation
pdf_to_semantic_markdown("data/Bitcoin/bitcoin.pdf", "data/Bitcoin/Bitcoin_markdown.md", img_dir="data/Bitcoin/images_tmp")

Extraction du PDF...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0002-02.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0002-07.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0003-03.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0004-07.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0005-02.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0005-06.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0006-02.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0006-11.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0007-04.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0007-06.png via Ollama...
Analyse de l'image : data/Bitcoin/images_tmp/bitcoin.pdf-0007-08.png via Ollama...
Terminé !


In [59]:
pdf_path = "data/TerraLuna/TerraLunaWP.pdf"
output_md = "data/TerraLuna/test/TerraLuna_markdown.md"
img_dir = "data/TerraLuna/images_tmp"

Path(img_dir).mkdir(parents=True, exist_ok=True)
Path(output_md).parent.mkdir(parents=True, exist_ok=True)

# 1. Extraction avec sauvegarde physique des images
print("Extraction du PDF...")
md_text = pymupdf4llm.to_markdown(
    pdf_path,
    write_images=True,
    image_path=img_dir,
    image_format="png",
    force_text=False,
)

img_pattern = re.compile(r'!\[.*?\]\((.*?)\)')
no_img_md_text = re.sub(img_pattern, '', md_text)


# 4. Sauvegarde du fichier final
Path(output_md).write_text(no_img_md_text, encoding="utf-8",)

print("Terminé !")

Extraction du PDF...
Terminé !


In [60]:
from pathlib import Path

img_dir = Path("data/TerraLuna/images_tmp")

pdf_size = Path(pdf_path).stat().st_size
total_size_img = sum(f.stat().st_size for f in img_dir.iterdir() if f.is_file())
size_text = len(no_img_md_text.encode('utf-8'))

print(f"Taille totale images : {total_size_img} octets ({total_size_img/1024:.1f} Ko)")
print(f"Poids du Markdown final : {size_text} octets ({size_text/1024:.1f} Ko)")
print(f"Poids du PDF original : {pdf_size} octets ({pdf_size/1024:.1f} Ko)")

Taille totale images : 324780 octets (317.2 Ko)
Poids du Markdown final : 31507 octets (30.8 Ko)
Poids du PDF original : 250355 octets (244.5 Ko)


In [ ]:
number_of_words = len(no_img_md_text.split())
print(f"Nombre de mots dans le Markdown final : {number_of_words}")

Nombre de mots dans le Markdown final : 5003


In [62]:
import textstat

print(f"Le Gunning Fog Index du Markdown final est de : {textstat.gunning_fog(no_img_md_text):.2f}")

Le Gunning Fog Index du Markdown final est de : 15.80


In [63]:
import nltk
import torch
from transformers import pipeline
from PyPDF2 import PdfReader  # ou pdfplumber selon ton PDF

nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize

# ─── 1. Lecture du whitepaper ───────────────────────────────────────────────

def read_pdf(path: str) -> str:
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

# ─── 2. Découpage en chunks cohérents ───────────────────────────────────────

def chunk_text(text: str, max_tokens: int = 400, overlap: int = 50) -> list[str]:
    """
    Découpe par phrases pour ne pas couper au milieu d'une idée.
    overlap = nombre de tokens communs entre chunks consécutifs (contexte).
    """
    sentences = sent_tokenize(text)
    chunks, current, current_len = [], [], 0

    for sentence in sentences:
        token_count = len(sentence.split())  # approximation rapide

        if current_len + token_count > max_tokens and current:
            chunks.append(" ".join(current))
            # Garder les dernières phrases pour l'overlap
            overlap_sents = []
            overlap_len = 0
            for s in reversed(current):
                if overlap_len + len(s.split()) <= overlap:
                    overlap_sents.insert(0, s)
                    overlap_len += len(s.split())
                else:
                    break
            current, current_len = overlap_sents, overlap_len

        current.append(sentence)
        current_len += token_count

    if current:
        chunks.append(" ".join(current))

    return chunks

# ─── 3. Pipeline de sentiment ────────────────────────────────────────────────

# Modèle financier/crypto : bien meilleur que généraliste
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",   # entraîné sur textes financiers
    tokenizer="ProsusAI/finbert",
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=512,
) #type: ignore

# ─── 4. Analyse et agrégation ────────────────────────────────────────────────

def analyze_document(text: str) -> dict:
    chunks = chunk_text(text)
    print(f"→ {len(chunks)} chunks générés")

    all_results = []

    for i, chunk in enumerate(chunks):
        try:
            # Récupérer les scores des 3 labels (pas juste le dominant)
            out = sentiment_pipeline(chunk, top_k=3)  # top_k=None pour tous les labels
            scores = {item["label"].lower(): round(item["score"], 3) for item in out}
            all_results.append({"chunk": i + 1, "scores": scores, "preview": chunk[:80]})
        except Exception as e:
            print(f"Chunk {i+1} ignoré : {e}")

    # Moyenne des scores par label (plus juste que la somme)
    avg = {}
    for label in ["positive", "negative", "neutral"]:
        avg[label] = round(
            sum(r["scores"].get(label, 0) for r in all_results) / len(all_results), 3
        )

    dominant = max(avg, key=avg.get)

    # Chunks les plus "chargés" émotionnellement
    most_positive = max(all_results, key=lambda x: x["scores"].get("positive", 0))
    most_negative = max(all_results, key=lambda x: x["scores"].get("negative", 0))

    return {
        "dominant_sentiment": dominant,
        "avg_scores": avg,
        "num_chunks": len(chunks),
        "most_positive_chunk": most_positive,
        "most_negative_chunk": most_negative,
        "details": all_results,
    }
# ─── 5. Analyse par section ──────────────────────────────────────────────────

SECTIONS = ["abstract", "introduction", "tokenomics", "roadmap", "risks", "conclusion"]

def analyze_by_section(text: str) -> dict:
    """Segmente le doc par sections connues des whitepapers."""
    text_lower = text.lower()
    section_results = {}

    for i, section in enumerate(SECTIONS):
        start = text_lower.find(section)
        if start == -1:
            continue
        # Fin = début de la section suivante
        end = len(text)
        for next_section in SECTIONS[i + 1:]:
            pos = text_lower.find(next_section, start + 1)
            if pos != -1:
                end = pos
                break

        section_text = text[start:end]
        result = analyze_document(section_text)
        section_results[section] = result["dominant_sentiment"], result["scores"]

    return section_results


[nltk_data] Downloading package punkt_tab to /home/hassan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [70]:
import re

def extract_sections_from_markdown(md_text: str) -> dict[str, str]:
    """
    Extrait les sections d'un document Markdown en se basant sur les titres.
    Retourne un dict {titre: contenu}.
    """
    # Trouver tous les titres et leur position
    pattern = re.compile(r'^(#{1,3})\s+(.+)$', re.MULTILINE)
    matches = list(pattern.finditer(md_text))

    if not matches:
        return {"full_document": md_text}

    sections = {}
    for i, match in enumerate(matches):
        level   = len(match.group(1))   # 1, 2 ou 3
        title   = match.group(2).strip()
        start   = match.end()
        end     = matches[i + 1].start() if i + 1 < len(matches) else len(md_text)
        content = md_text[start:end].strip()

        if content:  # ignorer les sections vides
            # Préfixer par le niveau pour garder la hiérarchie
            key = f"{'  ' * (level - 1)}{'#' * level} {title}"
            sections[key] = content

    return sections


def print_plan(sections: dict) -> None:
    """Affiche le plan du document."""
    print("=== PLAN DU DOCUMENT ===\n")
    for i, title in enumerate(sections.keys(), 1):
        words = len(sections[title].split())
        print(f"{i:>3}. {title}  ({words} mots)")

In [71]:
sections = extract_sections_from_markdown(no_img_md_text)
print_plan(sections)

=== PLAN DU DOCUMENT ===

  1.   ## Stability and Adoption  (11 mots)
  2.   ## **Abstract**  (136 mots)
  3.   ## **1 Introduction**  (466 mots)
  4.   ## **2 Multi-fiat peg monetary policy**  (105 mots)
  5.   ## **2.1 Defining stability against regional fiat currencies**  (277 mots)
  6.   ## **2.2 Measuring stability with miner oracles**  (292 mots)
  7.   ## **2.3 Achieving stability with consistent mining rewards**  (387 mots)
  8.   ## **2.4 Miners absorb short-term Terra volatility**  (467 mots)
  9.   ## **2.5 Miners are compensated with long-term stable rewards**  (612 mots)
 10.   ## **–** increase Luna burn  (733 mots)
 11.   ## **3 Growth-driven fiscal policy**  (1167 mots)
 12.   ## **4 Conclusion**  (206 mots)
 13.   ## **References**  (73 mots)


In [73]:
import re

def extract_major_sections(md_text: str) -> dict:
    """
    Extrait uniquement les grandes parties du document, même si tout est en '##'.
    Regroupe les sous-sections numérotées sous leur section parente.
    """
    # Capture tous les titres ## avec leur position
    pattern = re.compile(r'^##\s+(.+)$', re.MULTILINE)
    matches = list(pattern.finditer(md_text))

    if not matches:
        return {"Document entier": md_text.strip()}

    # Analyse des titres pour déterminer la hiérarchie réelle
    sections = {}
    current_major = None
    current_major_content = []
    start_last = matches[0].end()  # début du contenu après le premier titre

    for i, match in enumerate(matches):
        title_raw = match.group(1).strip()
        # Nettoyage : supprimer les éventuelles marques markdown dans le titre (**)
        title_clean = title_raw.strip('*').strip()

        # Définir si ce titre est une section principale ou une sous-section
        is_major = False
        # Sections principales reconnues : numéro simple (1,2,3...) ou noms spéciaux
        if re.match(r'^\d+\s', title_clean):  # ex: "1 Introduction", "2 Multi-fiat..."
            # C'est une section principale si le numéro n'est pas suivi d'un point (exclut 2.1)
            if not re.match(r'^\d+\.\d+\s', title_clean):
                is_major = True
        elif title_clean.lower() in ('abstract', 'conclusion', 'references'):
            is_major = True

        # Récupérer le contenu entre le titre précédent et celui-ci
        end_prev = matches[i-1].end() if i > 0 else 0
        content_chunk = md_text[start_last:match.start()].strip()

        if is_major:
            # Sauvegarder la section majeure précédente si elle existe
            if current_major is not None:
                full_content = "\n".join(current_major_content).strip()
                if full_content and len(full_content.split()) >= 30:
                    sections[current_major] = full_content

            # Commencer une nouvelle section majeure
            current_major = title_clean
            current_major_content = [content_chunk] if content_chunk else []
        else:
            # Sous-section : on l'ajoute au contenu de la section majeure courante
            # On peut aussi stocker le titre de la sous-section pour information, mais pas de nouvelle entrée
            if current_major is not None:
                current_major_content.append(f"### {title_clean}")  # marque la sous-section
                current_major_content.append(content_chunk)
            # Si on n'a pas encore de section majeure (ex: Stability and Adoption au début), on ignore

        start_last = match.end()

    # Traiter le dernier bloc (après le dernier titre)
    final_content = md_text[start_last:].strip()
    if current_major is not None:
        current_major_content.append(final_content)
        full_content = "\n".join(current_major_content).strip()
        if full_content and len(full_content.split()) >= 30:
            sections[current_major] = full_content

    return sections

major_sections = extract_major_sections(no_img_md_text)

# Affichage du nouveau plan
print("=== PLAN DES GRANDES PARTIES ===\n")
for i, (title, content) in enumerate(major_sections.items(), 1):
    print(f"{i:>3}. {title}  ({len(content.split())} mots)")

# Analyse de sentiment sur ces grandes parties seulement
results = {}
for title, content in major_sections.items():
    res = analyze_document(content)
    results[title] = {
        "dominant": res["dominant_sentiment"],
        "scores": res["avg_scores"],
        "words": len(content.split()),
    }
    dominant = res["dominant_sentiment"]
    scores = res["avg_scores"]
    print(f"\n{title}")
    print(f"  → {dominant.upper():10} | pos={scores['positive']:.2f}  neg={scores['negative']:.2f}  neu={scores['neutral']:.2f}")

=== PLAN DES GRANDES PARTIES ===

  1. 1 Introduction  (136 mots)
  2. 2 Multi-fiat peg monetary policy  (2650 mots)
  3. 3 Growth-driven fiscal policy  (733 mots)
  4. 4 Conclusion  (1167 mots)
  5. References  (279 mots)
→ 1 chunks générés

1 Introduction
  → NEUTRAL    | pos=0.40  neg=0.01  neu=0.60
→ 8 chunks générés

2 Multi-fiat peg monetary policy
  → NEUTRAL    | pos=0.09  neg=0.06  neu=0.86
→ 2 chunks générés

3 Growth-driven fiscal policy
  → NEUTRAL    | pos=0.12  neg=0.02  neu=0.85
→ 3 chunks générés

4 Conclusion
  → NEUTRAL    | pos=0.07  neg=0.02  neu=0.91
→ 1 chunks générés

References
  → NEUTRAL    | pos=0.14  neg=0.01  neu=0.86


In [77]:
def analyze_by_major_sections(md_text: str) -> dict:
    sections = extract_major_sections(md_text)

    print(f"→ {len(sections)} sections détectées\n")
    results = {}

    for title, content in sections.items():
        # Les sections trop courtes ne sont pas fiables
        if len(content.split()) < 30:
            continue

        result = analyze_document(content)
        results[title] = {
            "dominant": result["dominant_sentiment"],
            "scores":   result["avg_scores"],
            "words":    len(content.split()),
        }

        dominant = result["dominant_sentiment"]
        scores   = result["avg_scores"]
        print(f"{title}")
        print(f"  → {dominant.upper():10} | pos={scores['positive']:.2f}  neg={scores['negative']:.2f}  neu={scores['neutral']:.2f}\n")

    return results

# Lancement
section_analysis = analyze_by_major_sections(no_img_md_text)

→ 5 sections détectées

→ 1 chunks générés
1 Introduction
  → NEUTRAL    | pos=0.40  neg=0.01  neu=0.60

→ 8 chunks générés
2 Multi-fiat peg monetary policy
  → NEUTRAL    | pos=0.09  neg=0.06  neu=0.86

→ 2 chunks générés
3 Growth-driven fiscal policy
  → NEUTRAL    | pos=0.12  neg=0.02  neu=0.85

→ 3 chunks générés
4 Conclusion
  → NEUTRAL    | pos=0.07  neg=0.02  neu=0.91

→ 1 chunks générés
References
  → NEUTRAL    | pos=0.14  neg=0.01  neu=0.86



In [ ]:
def analyze_by_markdown_sections(md_text: str) -> dict:
    sections = extract_sections_from_markdown(md_text)

    print(f"→ {len(sections)} sections détectées\n")
    results = {}

    for title, content in sections.items():
        # Les sections trop courtes ne sont pas fiables
        if len(content.split()) < 30:
            continue

        result = analyze_document(content)
        results[title] = {
            "dominant": result["dominant_sentiment"],
            "scores":   result["avg_scores"],
            "words":    len(content.split()),
        }

        dominant = result["dominant_sentiment"]
        scores   = result["avg_scores"]
        print(f"{title}")
        print(f"  → {dominant.upper():10} | pos={scores['positive']:.2f}  neg={scores['negative']:.2f}  neu={scores['neutral']:.2f}\n")

    return results

# Lancement
section_analysis = analyze_by_markdown_sections(no_img_md_text)

→ 13 sections détectées

→ 1 chunks générés
  ## **Abstract**
  → NEUTRAL    | pos=0.40  neg=0.01  neu=0.60

→ 2 chunks générés
  ## **1 Introduction**
  → NEUTRAL    | pos=0.12  neg=0.05  neu=0.83

→ 1 chunks générés
  ## **2 Multi-fiat peg monetary policy**
  → NEUTRAL    | pos=0.16  neg=0.03  neu=0.81

→ 1 chunks générés
  ## **2.1 Defining stability against regional fiat currencies**
  → NEUTRAL    | pos=0.13  neg=0.03  neu=0.84

→ 1 chunks générés
  ## **2.2 Measuring stability with miner oracles**
  → NEUTRAL    | pos=0.05  neg=0.07  neu=0.88

→ 1 chunks générés
  ## **2.3 Achieving stability with consistent mining rewards**
  → NEUTRAL    | pos=0.08  neg=0.02  neu=0.90

→ 2 chunks générés
  ## **2.4 Miners absorb short-term Terra volatility**
  → NEUTRAL    | pos=0.11  neg=0.02  neu=0.87

→ 2 chunks générés
  ## **2.5 Miners are compensated with long-term stable rewards**
  → NEUTRAL    | pos=0.07  neg=0.08  neu=0.84

→ 2 chunks générés
  ## **–** increase Luna burn
  → NEUTRAL 

In [2]:
import os
import requests
import re
from bs4 import BeautifulSoup
import urllib.parse
import time

def download_crypto_whitepaper(gitbook_url, project_name="crypto_project"):
    print(f"--- Analyse du Whitepaper pour {project_name} ---")
    
    # 1. Récupérer la structure de la page d'accueil
    try:
        response = requests.get(gitbook_url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
    except Exception as e:
        print(f"Impossible d'accéder à l'URL : {e}")
        return

    # Extraire tous les liens du menu ou de la page
    links = []
    base_domain = urllib.parse.urlparse(gitbook_url).netloc
    
    for a in soup.find_all('a', href=True):
        href = a['href']
        full_url = urllib.parse.urljoin(gitbook_url, href).split('#')[0]
        
        # On ne garde que les pages qui restent sur le GitBook du projet
        if urllib.parse.urlparse(full_url).netloc == base_domain:
            if full_url not in links:
                links.append(full_url)
                
    # S'assurer que l'URL racine est en premier
    if gitbook_url in links:
        links.remove(gitbook_url)
    links.insert(0, gitbook_url)

    print(f"Trouvé : {len(links)} sections à télécharger.")
    
    # 2. Boucler et télécharger via le proxy Jina AI
    full_markdown_content = f"# Whitepaper: {project_name.upper()}\nSource: {gitbook_url}\n\n"
    
    for i, url in enumerate(links):
        print(f"[{i+1}/{len(links)}] Téléchargement de : {url}")
        
        # On passe par le proxy de Jina pour avoir un Markdown propre sans code HTML résiduel
        jina_url = f"https://r.jina.ai/{url}"
        
        try:
            # Jina gère le JavaScript des GitBooks modernes à notre place
            res = requests.get(jina_url, timeout=15)
            if res.status_code == 200:
                # Nettoyer un peu les métadonnées Jina en haut de fichier si nécessaire
                content = res.text
                full_markdown_content += f"\n\n<!-- Début Section: {url} -->\n\n" + content
            else:
                print(f"Échec sur cette section (Code {res.status_code})")
        except Exception as e:
            print(f"Erreur de connexion : {e}")
            
        # Petit sleep de courtoisie pour l'API Jina
        time.sleep(0.5)

    # 3. Sauvegarder le livret complet
    os.makedirs("crypto_whitepapers", exist_ok=True)
    output_filename = f"crypto_whitepapers/{project_name}_complete.md"
    
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(full_markdown_content)
        
    print(f"Terminé ! Whitepaper complet sauvegardé dans : {output_filename}")

# --- Test direct ---
if __name__ == "__main__":
    # Remplace par le GitBook du token que tu veux analyser
    URL_CRYPTO = "https://hyperliquid.gitbook.io/hyperliquid-docs" 
    download_crypto_whitepaper(URL_CRYPTO, project_name="uniswap")

--- Analyse du Whitepaper pour uniswap ---
Trouvé : 21 sections à télécharger.
[1/21] Téléchargement de : https://hyperliquid.gitbook.io/hyperliquid-docs
[2/21] Téléchargement de : https://hyperliquid.gitbook.io/hyperliquid-docs/builder-tools
[3/21] Téléchargement de : https://hyperliquid.gitbook.io/hyperliquid-docs/support
[4/21] Téléchargement de : https://hyperliquid.gitbook.io/hyperliquid-docs/about-hyperliquid/hyperliquid-101-for-non-crypto-audiences
[5/21] Téléchargement de : https://hyperliquid.gitbook.io/hyperliquid-docs/about-hyperliquid/core-contributors
[6/21] Téléchargement de : https://hyperliquid.gitbook.io/hyperliquid-docs/onboarding
[7/21] Téléchargement de : https://hyperliquid.gitbook.io/hyperliquid-docs/hypercore
[8/21] Téléchargement de : https://hyperliquid.gitbook.io/hyperliquid-docs/hyperevm
[9/21] Téléchargement de : https://hyperliquid.gitbook.io/hyperliquid-docs/hyperliquid-improvement-proposals-hips
[10/21] Téléchargement de : https://hyperliquid.gitbook.io/h